# Step 1 — Direct Spark submission

This notebook is a **read-only participant guide** with no executable cells.
Read it in JupyterLab; use `STEP1-REMOTE.md` for the managed terminal launch
procedure. The existing `01_spark_basics.py` performs the analysis in your
assigned remote client through `spark_remote.py`.

## Before you start

Wait for organiser permission and confirmation of your assigned workspace and
prepared environment. The commands are documented, but **participant execution
is not yet validated**. Opening this guide or publishing its files does not
establish connectivity, capacity, isolation or cleanup.

Organiser-controlled container tests have passed on both Tier-2 VMs. That is
not validation of actual participant terminals or failure recovery. Do not run
the analysis script directly in JupyterLab, install PySpark/Java or packages,
guess connection settings, or manage Docker containers.

## Learning objectives

- Identify the driver, master, workers and executors.
- Explain how a CSV becomes a Spark DataFrame.
- Distinguish transformations from actions.
- Interpret selection, filtering and grouped counts.
- Find a bounded result and distinguish remote output from its local download.
- Understand session cleanup and when organiser inspection is required.


## 1. Locate your files and understand the dataset

In the assigned Spark workspace, locate this guide, `01_spark_basics.py` and `data/Bank_Personal_Loan_Modelling.csv`. Do not edit the supplied CSV.

Through `spark_remote.py analyse`, the script runs in the assigned remote client. It resolves its local input and output locations relative to its own file there, not the JupyterLab terminal's working directory. The prepared client provides the dataset link. Its first activity previews that CSV without contacting Spark; participants do not create the link themselves.

**Expected output:** column names and up to five records. The supplied snapshot contains 5,000 records and 14 columns. If files are missing, stop and report the missing path to the organiser.

Use the accompanying README and dataset documentation for provenance, licence, units and category definitions. Do not infer the meaning of a field from its numeric values alone. The existing dataset review reported 52 negative `Experience` values; discuss these rather than silently correcting them.

**Reflection:** why should a local file preview succeed before you attempt distributed processing?


## 2. Start the exercise using the managed launcher

Read `STEP1-REMOTE.md`. In the organiser-prepared JupyterLab terminal environment:

```bash
cd ~/Big-Data/Spark
# Submit only after organiser permission.
python3 spark_remote.py test
# After successful test completion and organiser approval:
python3 spark_remote.py analyse
```

`test` submits SparkPi. `analyse` uploads the existing `01_spark_basics.py` and
submits it using real `spark-submit` in your assigned client. It performs the
following analysis sections automatically; observe terminal output rather
than executing notebook cells. Do not run `python3 01_spark_basics.py` directly
in JupyterLab for this path.

**One active submission per group/client. Take turns.** Remote script and
preview filenames are shared, and the launcher has no concurrency lock.
Unique downloaded-result directories do not prevent remote job collisions.

### Execution path

- **JupyterHub terminal:** SSH/SFTP transport using organiser-prepared Python
  with Paramiko; it does not host the Spark driver.
- **Driver:** the application's coordinating process in your assigned client
  container. It plans work and collects the bounded preview.
- **Master:** the private standalone service coordinating resource allocation.
- **Worker:** the assigned VM's service launching application executors.
- **Executors:** processes executing tasks on data partitions.

The driver is not this Markdown notebook. Executors must connect back to it;
reaching the master alone is insufficient. This is a VM-local one-worker pilot,
not a Spark cluster spanning workshop VMs.

### Organiser-provided connection and data

The launcher reads your group's `login_key` and `known_hosts` under
`~/.ssh/spark-direct/` and derives the endpoint from its single pinned host
record. Do not transfer keys manually or weaken host-key checking.

The prepared client supplies `SPARK_MASTER_URL`, `SPARK_DATA_PATH`,
`SPARK_DRIVER_HOST` and the local-preview dataset link. Optional fixed callback
ports remain organiser configuration. Do not export guessed settings in
JupyterLab. Local preview and distributed input paths identify the same
supplied snapshot even though they may differ.

Commands limit executor cores to one and driver/executor memory to 512 MB each.
Remote timeouts are 120 seconds for SparkPi and 300 seconds for analysis.
Timeout or connection closure is not proof of remote cleanup.

**Expected output:** SparkPi output for the test; Spark version and master URL
for analysis. Check `Remote exit status:`. Session creation or a successful
SparkPi run alone does not prove bank-data processing and retrieval will work.


## 3. Observe the schema and full input count

The script reads the CSV with a header and inferred column types, prints the schema, displays five records and counts the full input.

A **DataFrame** is tabular data with named, typed columns. Schema inference inspects the data; `show` and `count` request processing results.

**Expected result for the supplied snapshot:** **5,000 records**. Check the inferred types of `Age`, `Income` and `Family`. If the count differs, report it rather than editing the input to match.

**Reflection:** which fields are measurements, categories or counts? Use the source documentation to support your answer.


## 4. Observe the strict age filter

The script selects `Age`, `Income` and `Family`, then retains records where **Age > 30**. Records with age exactly 30 are excluded.

Selection and filtering are **transformations**: they build a plan. Displaying and counting are **actions**: they request execution. This deferred execution is called lazy evaluation.

**Expected result:** **4,376 matching records**, with up to ten displayed rows and only the three selected columns. These counts were also observed in the organiser-controlled Tier-2 Livy tests using the same snapshot.

The preview is not the full filtered dataset. No explicit row ordering is applied, so preview rows may differ between runs.

**Reflection:** why are the displayed row count and the matching-record count different?


## 5. Observe the Personal Loan category counts

The script groups records by the raw `Personal Loan` value, counts each group and sorts the summary by category value.

**Expected result:** category **0: 4,520 records**; category **1: 480 records**. The counts sum to the complete input count of 5,000.

Check the dataset documentation before assigning meanings to these codes. This exercise describes the data; it does not train a predictive model or demonstrate causation.

A grouped result alone does not establish execution across multiple workers. The validated pilot uses one Spark worker within each VM's assigned runtime.

**Reflection:** which category is more frequent, and why could that matter in a later modelling task?


## 6. Locate the bounded preview

The analysis collects **at most ten filtered rows** to its remote driver and
writes `outputs/filtered_preview.csv` beside the script in the assigned client.

The CSV has the `Age,Income,Family` header and up to ten data rows. An empty
filter result produces a header-only file. This is a bounded preview, not
Spark's distributed CSV output directory. The source CSV is not modified;
do not remove the collection limit.

After successful remote analysis, the launcher downloads the preview and prints
`Downloaded preview:` with its local path:

```text
outputs/remote-<unique-id>/filtered_preview.csv
```

For the default script, that directory is inside `~/Big-Data/Spark`. Open the
printed download path in JupyterLab, not the remote `Saved:` path. Each download
gets a new local directory. Remote reruns still replace the shared remote
preview; new local directories do not make concurrent submissions safe.

If retrieval fails after `Remote exit status: 0`, report the retrieval error.
Do not automatically resubmit analysis or assume an existing remote preview
belongs to your run.

**Reflection:** why is collecting ten rows safer than collecting an arbitrarily
large result, and why is output retrieval a separate success check?


## 7. Finish and check cleanup

Once it obtains a Spark session, the analysis attempts `spark.stop()` in a
`finally` block after successful analysis or an analysis error. It does not stop
shared runtime containers. Cleanup can fail or be interrupted.

**Normal analysis messages:** `Spark session stopped.` followed by
`Analysis completed successfully.` The launcher also reports
`Remote exit status: 0` and, after a successful download, `Downloaded preview:`.
Check execution and retrieval separately.

The launcher is not a remote process-supervision system. A timeout, SSH
connection closure, closed terminal or Ctrl-C does not prove driver termination.
After an interruption or uncertain cleanup, stop and ask the organiser to
inspect the remote application before retrying or allowing another group member
to submit. Never stop shared containers yourself.

The organiser must validate actual participant execution, worker/application
evidence, result retrieval and cleanup. The menu's automated participant
validation and non-destructive stop are not implemented yet. Publication and
organiser-only test success do not establish readiness.

Complete Step 1 and resolve cleanup before proceeding. Step 2 requires its own
separately approved launch path; this Stage 1 workflow does not enable it.


## Troubleshooting and completion checklist

For missing credentials, host-key mismatch, missing Paramiko, dataset errors,
connection failures, timeouts or cleanup uncertainty: **stop and report the
command, failing stage, error, remote exit status and relevant printed paths**.
Include an application ID when available, but never private-key content or
credential payloads.

Do not change ports, invent settings, replace host-key pins, install software,
start/stop shared services or repeatedly submit jobs.

- [ ] I read `STEP1-REMOTE.md` and used only its authorised launcher commands.
- [ ] I ran one submission at a time for my assigned group/client.
- [ ] I distinguished the JupyterHub transport from the remote Spark driver.
- [ ] I identified the master, worker and executor roles.
- [ ] I checked the schema and 5,000-record input count.
- [ ] I explained the strict filter and 4,376 matching records.
- [ ] I checked the category counts and their sum.
- [ ] I located the unique local download and understood remote overwrite behaviour.
- [ ] I checked completion/retrieval and reported cleanup uncertainty before continuing.

Treat `docs/spark-DA.pdf` as conceptual/historical background, not managed-workshop
installation instructions. Older guidance asking you to execute notebook cells
does not apply to this Markdown-only guide.
